# 04 · Modelling, Evaluation & Explainability
Training is done by [`src/train.py`](../src/train.py) (re-run with `python src/train.py`, ~10 min). This notebook explains the design and analyses the saved results in `models/metrics.json`.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60); pd.set_option("display.float_format", "{:,.3f}".format)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)


## 1. Experimental design
* **Time-based split:** train = orders purchased **2017-01 → 2018-03**, test = **2018-04 → 2018-08**. A random split would leak future patterns and overstate performance; this mimics deploying the model on new orders.
* **Model selection** by **3-fold stratified cross-validation on the training period only**; the test period is touched once at the end.
* **Class imbalance:** late ≈ 7%, bad review ≈ 22%. We use `class_weight="balanced"` for linear/tree models, PR-AUC as the main metric (accuracy is misleading: predicting "never late" gives 95% accuracy) and **tune the decision threshold** to maximise F1 on out-of-fold training predictions.
* **Candidates:** Gaussian Naive Bayes (baseline) → Logistic Regression → Decision Tree → Random Forest → XGBoost → XGBoost tuned with `RandomizedSearchCV`.
* Every model is a single scikit-learn `Pipeline` (log-transform + scaling + one-hot encoding + classifier), so the exact same preprocessing runs in the web app.

In [ ]:
import json, joblib
metrics = json.loads((ROOT / "models/metrics.json").read_text())
for t, m in metrics.items():
    print(f"{m['title']}: train {m['n_train']:,} ({m['train_period']}), test {m['n_test']:,} ({m['test_period']}), "
          f"positive rate train {m['pos_rate_train']:.1%} / test {m['pos_rate_test']:.1%}")

## 2. Model comparison (cross-validation on training period)

In [ ]:
for t, m in metrics.items():
    cv = pd.DataFrame(m["cv"]).set_index("model")
    print(m["title"]); display(cv[["roc_auc", "pr_auc", "f1", "precision", "recall"]].style.highlight_max(color="#c6efce").format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (t, m) in zip(axes, metrics.items()):
    cv = pd.DataFrame(m["cv"]).set_index("model")
    cv[["roc_auc", "pr_auc", "f1"]].plot.barh(ax=ax, xerr=cv[["roc_auc_std", "pr_auc_std", "f1_std"]].values.T)
    ax.axvline(m["pos_rate_train"], ls="--", c="grey"); ax.set_title(m["title"] + " — CV"); ax.set_xlim(0, 1)
plt.tight_layout(); plt.savefig(FIG / "model_comparison.png"); plt.show()

The dashed line is the PR-AUC of a random classifier (= positive rate). Every model beats it; **Naive Bayes is the weakest** (independence assumption violated), tree ensembles are best.

## 3. Final model on the untouched test period

In [ ]:
rows = []
for t, m in metrics.items():
    rows.append({"task": m["title"], "best model": m["best_model"], "threshold": m["threshold"], **m["test"]})
pd.DataFrame(rows).set_index("task")

In [ ]:
for t, m in metrics.items():
    print(m["title"]); display(pd.DataFrame(m["test_all_models_at_0.5"]).set_index("model").style.format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for row, (t, m) in zip(axes, metrics.items()):
    cm = np.array(m["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=row[0], cbar=False,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    row[0].set_title(f"{m['title']}\nconfusion matrix @ thr={m['threshold']:.2f}")
    row[1].plot(m["roc_curve"]["fpr"], m["roc_curve"]["tpr"]); row[1].plot([0, 1], [0, 1], "k--")
    row[1].set(title=f"ROC (AUC = {m['test']['roc_auc']:.3f})", xlabel="FPR", ylabel="TPR")
    row[2].plot(m["pr_curve"]["recall"], m["pr_curve"]["precision"]); row[2].axhline(m["pos_rate_test"], ls="--", c="grey")
    row[2].set(title=f"Precision-Recall (AP = {m['test']['pr_auc']:.3f})", xlabel="Recall", ylabel="Precision")
plt.tight_layout(); plt.savefig(FIG / "test_evaluation.png"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (t, m) in zip(axes, metrics.items()):
    c = m["calibration"]; ax.plot(c["mean_predicted"], c["fraction_positive"], "o-"); ax.plot([0, 1], [0, 1], "k--")
    ax.set(title=f"Calibration — {t}", xlabel="predicted probability", ylabel="observed frequency")
plt.tight_layout(); plt.show()

## 4. What drives the predictions? (permutation importance on test set)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (t, m) in zip(axes, metrics.items()):
    imp = pd.DataFrame(m["importance"]).head(12).iloc[::-1]
    ax.barh(imp.feature, imp.importance, xerr=imp["std"], color="C2"); ax.set_title(f"Permutation importance — {t}\n(drop in PR-AUC when shuffled)")
plt.tight_layout(); plt.savefig(FIG / "importance.png"); plt.show()

## 5. Using a saved model on a new situation

In [ ]:
from src.features import add_derived
bundle = joblib.load(ROOT / "models/satisfaction.joblib")
order = pd.DataFrame([bundle["defaults"]])
for scenario, actual in [("arrived 5 days early", bundle["defaults"]["estimated_delivery_days"] - 5),
                         ("arrived 10 days late", bundle["defaults"]["estimated_delivery_days"] + 10)]:
    x = add_derived(order.assign(actual_delivery_days=actual))[bundle["features"]]
    p = bundle["pipeline"].predict_proba(x)[0, 1]
    print(f"Typical order {scenario:22s} → P(bad review) = {p:.1%}  → {'BAD' if p >= bundle['threshold'] else 'good'}")

## 6. Conclusions & limitations
* **Satisfaction model** is strong: delivery delay, seller history and basket size explain most bad reviews — operations can proactively contact customers whose order is predicted to produce a bad review.
* **Late-delivery model** is harder (rare event + drift between periods: 7.5% late in train vs 4% in test) but still ranks risky orders far better than chance — useful for flagging orders for priority shipping.
* Limitations: no information about carriers or weather; review text not used; concept drift means the model should be retrained monthly.
* Future work: NLP sentiment on Portuguese review text, carrier-level features, survival models for delivery time.